## Preliminary Dataset Cleaning: Herculean

### 1. Dataset Import and Overview
URL: https://huggingface.co/datasets/TheFinAI/Herculean#loading

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "prices.parquet").exists():
    ROOT = ROOT / "Evelyn_Herculean"
DATA = ROOT / "data"
if not (DATA / "prices.parquet").exists():
    raise FileNotFoundError("First run download_herculean.py in the herculean directory, then open the Notebook from the project root or the herculean directory.")

prices = pd.read_parquet(DATA / "prices.parquet")
news = pd.read_parquet(DATA / "news.parquet")
filings = pd.read_parquet(DATA / "filings.parquet")

In [2]:
# Analysis scope
analysis_symbols = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"
]

start_date = "2024-12-01"
end_date = "2026-03-31"

pd.DataFrame({
    "item": ["Question", "Stocks", "Start date", "End date"],
    "description": [
        "How does company news relate to subsequent returns and volume?",
        ", ".join(analysis_symbols),
        start_date,
        end_date,
    ],
})

,item,description
0,Question,How does company news relate to subsequent ret...
1,Stocks,"AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA"
2,Start date,2024-12-01
3,End date,2026-03-31


In [3]:
# dataset Overview
pd.DataFrame({
    "table": ["prices", "news", "filings"],
    "rows": [len(prices), len(news), len(filings)],
    "columns": [len(prices.columns), len(news.columns), len(filings.columns)],
})

,table,rows,columns
0,prices,2656,9
1,news,3888,4
2,filings,73,6


In [4]:

display(prices.head(7))

display(news.head(7))

display(filings.head(7))

,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600
5,6,AAPL,2024-12-09,241.830002,247.240005,241.750000,246.750000,245.409409,44649200
6,7,AAPL,2024-12-10,246.889999,248.210007,245.339996,247.770004,246.423859,36914800


,id,symbol,date,highlights
0,1,AAPL,2024-12-01,"Apple’s December 1, 2024 coverage centers on i..."
1,2,AAPL,2024-12-02,"The December 2, 2024 Apple coverage portrays a..."
2,3,AAPL,2024-12-03,"demonstrated strong financial performance, wit..."
3,4,AAPL,2024-12-04,Across the December 3 coverage compiled for Ap...
4,5,AAPL,2024-12-05,- The Shacknews report highlights Apple’s firs...
5,6,AAPL,2024-12-06,"Apple Inc. is portrayed in the December 6, 202..."
6,7,AAPL,2024-12-07,"Three articles published around December 6–7, ..."


,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
1,2,AAPL,2024-05-02,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
2,3,AAPL,2024-08-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
3,4,AAPL,2024-11-01,10-K,Item 7. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
4,5,AAPL,2025-01-31,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
5,6,AAPL,2025-05-02,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
6,7,AAPL,2025-08-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...


In [5]:
tables = {
    "prices": prices,
    "news": news,
    "filings": filings,
}

for name, frame in tables.items():
    print(name)

    display(pd.DataFrame({
        "rows": [len(frame)],
        "unique_ids": [frame["id"].nunique()],
        "unique_symbol_date_keys": [
            len(frame[["symbol", "date"]].drop_duplicates())
        ],
        "missing_key_rows": [
            frame[["symbol", "date"]].isna().any(axis=1).sum()
        ],
    }))

# Additional candidate key for filings
print(
    "Repeated filing keys:",
    filings.duplicated(["symbol", "date", "document_type"]).sum()
)

prices


,rows,unique_ids,unique_symbol_date_keys,missing_key_rows
0,2656,2656,2656,0


news


,rows,unique_ids,unique_symbol_date_keys,missing_key_rows
0,3888,3888,3888,0


filings


,rows,unique_ids,unique_symbol_date_keys,missing_key_rows
0,73,73,73,0


Repeated filing keys: 0


### 4. Inspect columns and types



#### Original-data baseline


In [6]:
# Create a summary without modifying prices, news, or filings
table_map = {
    "prices": prices,
    "news": news,
    "filings": filings,
}

overview_rows = []

for table_name, frame in table_map.items():
    dates = pd.to_datetime(frame["date"], errors="raise")

    overview_rows.append({
        "Table": table_name,
        "Rows": len(frame),
        "Columns": len(frame.columns),
        "Observed date coverage": (
            f"{dates.min().date()}–{dates.max().date()}"
        ),
    })

overview = pd.DataFrame(overview_rows)
overview

,Table,Rows,Columns,Observed date coverage
0,prices,2656,9,2024-12-02–2026-03-31
1,news,3888,4,2024-12-01–2026-03-31
2,filings,73,6,2024-01-16–2026-03-25


In [7]:
# Preserve the loaded originals
raw_tables = {
    name: frame.copy(deep=True)
    for name, frame in tables.items()
}

# Make changes only to these working copies
cleaned_tables = {
    name: frame.copy(deep=True)
    for name, frame in tables.items()
}

In [8]:
import hashlib

source_hashes_before = {}

for name in tables:
    path = DATA / f"{name}.parquet"
    source_hashes_before[name] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

pd.Series(source_hashes_before, name="original_file_hash")

prices     0001b983f1a302e2f7a2d2e6a6c407f44127de77e981b0...
news       9156b73a5c417057104abe9eeb0d9b3ea0dceea66260e5...
filings    2d9a93926e52d4e2f5f18e13d68fbdd60e14282f94df4e...
Name: original_file_hash, dtype: str

In [9]:
# Company-level summary
pd.DataFrame({
    "prices": prices.groupby("symbol").size(),
    "news": news.groupby("symbol").size(),
    "filings": filings.groupby("symbol").size(),
}).fillna(0).astype(int)

,prices,news,filings
symbol,,,
AAPL,332,486,9
ADBE,332,486,10
AMZN,332,486,9
GOOGL,332,486,9
META,332,486,9
MSFT,332,486,9
NVDA,332,486,9
TSLA,332,486,9



There are **0 exact duplicate rows, 0 repeated IDs, and 0 repeated company/date keys** in each source table, counting occurrences beyond the first. Filings also have 0 repeated `symbol` + `date` + `document_type` keys. These are baseline checks, not reasons to remove records or guarantees about future snapshots.

Each table contains the same eight labels: `AAPL`, `ADBE`, `AMZN`, `GOOGL`, `META`, `MSFT`, `NVDA`, and `TSLA`. Prices have 332 rows per company; news has 486; filings have 10 for ADBE and 9 for each other company. Symbols have no outer whitespace. Filing categories are `10-K` (22) and `10-Q` (51), also with no outer whitespace. 



#### Column inventory and intended handling


Inspect data type, missing value, and cleaning methods

In [10]:
for name, frame in raw_tables.items():
    print(name)

    display(pd.DataFrame({
        "dtype": frame.dtypes,
        "missing": frame.isna().sum(),
        "unique_values": frame.nunique(),
    }))

display(prices.describe())

prices


,dtype,missing,unique_values
id,int32,0,2656
symbol,str,0,8
date,object,0,332
open,float64,0,2556
high,float64,0,2562
low,float64,0,2562
close,float64,0,2579
adj_close,float64,0,2626
volume,int64,0,2654


news


,dtype,missing,unique_values
id,int32,0,3888
symbol,str,0,8
date,object,0,486
highlights,str,0,3888


filings


,dtype,missing,unique_values
id,int32,0,73
symbol,str,0,8
date,object,0,57
document_type,str,0,2
mda_content,str,0,73
risk_content,str,0,66


,id,open,high,low,close,adj_close,volume
count,2656.000000,2656.000000,2656.000000,2656.000000,2656.000000,2656.000000,2.656000e+03
mean,1328.500000,336.990791,341.288332,332.383935,336.871370,336.242330,6.003645e+07
std,766.865481,161.598139,163.212187,159.602522,161.338599,160.832188,7.165307e+07
min,1.000000,87.459999,99.440002,86.620003,94.309998,94.287781,1.166500e+06
25%,664.750000,209.545002,211.867500,206.967503,209.642494,209.113598,1.708360e+07
50%,1328.500000,287.915009,293.915009,282.240005,287.850006,287.850006,3.618155e+07
75%,1992.250000,433.405006,439.732506,426.514999,433.387505,432.083717,6.831292e+07
max,2656.000000,791.150024,796.250000,780.820007,790.000000,788.148987,8.188309e+08


In [11]:
text_columns = {
    "prices": ["symbol"],
    "news": ["symbol", "highlights"],
    "filings": [
        "symbol", "document_type", "mda_content", "risk_content"
    ],
}

text_checks = []

for name, columns in text_columns.items():
    for column in columns:
        text = raw_tables[name][column]

        text_checks.append({
            "table": name,
            "column": column,
            "outer_whitespace": (
                text.notna() & text.ne(text.str.strip())
            ).sum(),
            "blank_strings": text.str.strip().eq("").sum(),
        })

display(pd.DataFrame(text_checks))
display(filings["document_type"].value_counts(dropna=False))

,table,column,outer_whitespace,blank_strings
0,prices,symbol,0,0
1,news,symbol,0,0
2,news,highlights,0,0
3,filings,symbol,0,0
4,filings,document_type,0,0
5,filings,mda_content,73,0
6,filings,risk_content,73,0


document_type
10-Q    51
10-K    22
Name: count, dtype: int64


#### Date-conversion rule and measured trial results


In [12]:
for name, frame in raw_tables.items():
    parsed_dates = pd.to_datetime(frame["date"], errors="coerce")

    # Missing source dates and conversion failures are different
    failed = frame["date"].notna() & parsed_dates.isna()

    print(name, "date conversion failures:", failed.sum())
    display(frame.loc[failed, ["id", "symbol", "date"]])

    cleaned_tables[name]["date"] = pd.to_datetime(
        frame["date"], errors="raise"
    )

prices date conversion failures: 0


,id,symbol,date


news date conversion failures: 0


,id,symbol,date


filings date conversion failures: 0


,id,symbol,date


### Step 3 — Standardize text carefully (checklist 5)


decode financial report and strip the white spaces

In [13]:
text_columns = {
    "prices": ["symbol"],
    "news": ["symbol", "highlights"],
    "filings": [
        "symbol", "document_type", "mda_content", "risk_content"
    ],
}

text_checks = []

for name, columns in text_columns.items():
    for column in columns:
        text = raw_tables[name][column]

        text_checks.append({
            "table": name,
            "column": column,
            "outer_whitespace": (
                text.notna() & text.ne(text.str.strip())
            ).sum(),
            "blank_strings": text.str.strip().eq("").sum(),
        })

display(pd.DataFrame(text_checks))
display(filings["document_type"].value_counts(dropna=False))

,table,column,outer_whitespace,blank_strings
0,prices,symbol,0,0
1,news,symbol,0,0
2,news,highlights,0,0
3,filings,symbol,0,0
4,filings,document_type,0,0
5,filings,mda_content,73,0
6,filings,risk_content,73,0


document_type
10-Q    51
10-K    22
Name: count, dtype: int64


**5. Standardize text.** Inspect `symbol`, `document_type`, `highlights`, `mda_content`, and `risk_content` for whitespace, inconsistent labels, blank strings, and formatting artifacts. 

In [14]:
text_columns = {
    "prices": ["symbol"],
    "news": ["symbol", "highlights"],
    "filings": [
        "symbol", "document_type", "mda_content", "risk_content"
    ],
}

text_checks = []

for name, columns in text_columns.items():
    for column in columns:
        text = raw_tables[name][column]

        text_checks.append({
            "table": name,
            "column": column,
            "outer_whitespace": (
                text.notna() & text.ne(text.str.strip())
            ).sum(),
            "blank_strings": text.str.strip().eq("").sum(),
        })

display(pd.DataFrame(text_checks))
display(filings["document_type"].value_counts(dropna=False))

,table,column,outer_whitespace,blank_strings
0,prices,symbol,0,0
1,news,symbol,0,0
2,news,highlights,0,0
3,filings,symbol,0,0
4,filings,document_type,0,0
5,filings,mda_content,73,0
6,filings,risk_content,73,0


document_type
10-Q    51
10-K    22
Name: count, dtype: int64

In [15]:
import html

text_changes = []

for column in ["mda_content", "risk_content"]:
    original = raw_tables["filings"][column]
    decoded = original.map(html.unescape)
    cleaned = decoded.str.strip()

    cleaned_tables["filings"][column] = cleaned

    text_changes.append({
        "column": column,
        "decoded_cells": original.ne(decoded).sum(),
        "trimmed_cells": decoded.ne(cleaned).sum(),
        "total_changed_cells": original.ne(cleaned).sum(),
    })

    display(pd.DataFrame({
        "id": raw_tables["filings"]["id"],
        "before": original.str[:150],
        "after": cleaned.str[:150],
    }).head(1))

display(pd.DataFrame(text_changes))

,id,before,after
0,1,Item 2. Management&#8217;s Discussion and Ana...,Item 2. Management’s Discussion and Analysis o...


,id,before,after
0,1,Item 1A. Risk Factors \n\nThe Company&#8217;s...,Item 1A. Risk Factors \n\nThe Company’s busine...


,column,decoded_cells,trimmed_cells,total_changed_cells
0,mda_content,73,73,73
1,risk_content,68,73,73


### Step 4 — Investigate duplicates and missingness (checklist 6–7)



**6. Investigate duplicates.** Count exact duplicate rows, repeated `id` values, and repeated candidate business keys separately in each table. State whether each count measures duplicate groups, all involved rows, or extra rows beyond the first. Inspect conflicting records before deciding what to retain.


In [16]:
duplicate_checks = []

for stage, collection in [
    ("before", raw_tables),
    ("after", cleaned_tables),
]:
    for name, frame in collection.items():
        duplicate_checks.append({
            "stage": stage,
            "table": name,
            "duplicate_rows": frame.duplicated().sum(),
            "duplicate_ids": frame.duplicated(["id"]).sum(),
            "duplicate_symbol_date": (
                frame.duplicated(["symbol", "date"]).sum()
            ),
        })

display(pd.DataFrame(duplicate_checks))

,stage,table,duplicate_rows,duplicate_ids,duplicate_symbol_date
0,before,prices,0,0,0
1,before,news,0,0,0
2,before,filings,0,0,0
3,after,prices,0,0,0
4,after,news,0,0,0
5,after,filings,0,0,0



**7. Handle missing data.** Count missing values per column across the full tables, including blanks or placeholder strings you have verified actually mean “missing.” Distinguish absent source values, values made missing by conversion, and gaps introduced by a later merge.


In [17]:
for name in raw_tables:
    print(name)

    display(pd.DataFrame({
        "nulls_before": raw_tables[name].isna().sum(),
        "nulls_after": cleaned_tables[name].isna().sum(),
    }))

    columns = text_columns[name]

    display(pd.DataFrame({
        "blank_before": [
            raw_tables[name][c].str.strip().eq("").sum()
            for c in columns
        ],
        "blank_after": [
            cleaned_tables[name][c].str.strip().eq("").sum()
            for c in columns
        ],
    }, index=columns))

prices


,nulls_before,nulls_after
id,0,0
symbol,0,0
date,0,0
open,0,0
high,0,0
low,0,0
close,0,0
adj_close,0,0
volume,0,0


,blank_before,blank_after
symbol,0,0


news


,nulls_before,nulls_after
id,0,0
symbol,0,0
date,0,0
highlights,0,0


,blank_before,blank_after
symbol,0,0
highlights,0,0


filings


,nulls_before,nulls_after
id,0,0
symbol,0,0
date,0,0
document_type,0,0
mda_content,0,0
risk_content,0,0


,blank_before,blank_after
symbol,0,0
document_type,0,0
mda_content,0,0
risk_content,0,0


### Step 5 — Investigate suspicious records (checklist 8)


In [18]:
p = cleaned_tables["prices"].sort_values(["symbol", "date"])
price_columns = ["open", "high", "low", "close", "adj_close"]

price_flags = pd.DataFrame({
    "invalid_price": (
        p[price_columns].isna().any(axis=1)
        | p[price_columns].le(0).any(axis=1)
        | p[price_columns].isin(
            [float("inf"), float("-inf")]
        ).any(axis=1)
    ),
    "invalid_volume": (
        p["volume"].isna()
        | p["volume"].lt(0)
        | p["volume"].mod(1).ne(0)
    ),
    "invalid_OHLC_range": (
        p["low"].gt(p["high"])
        | p["open"].lt(p["low"])
        | p["open"].gt(p["high"])
        | p["close"].lt(p["low"])
        | p["close"].gt(p["high"])
    ),
})

daily_return = p.groupby("symbol")["adj_close"].pct_change(
    fill_method=None
)

# Screening threshold, not an automatic error/deletion rule
price_flags["large_return"] = daily_return.abs().gt(0.20)

display(price_flags.sum().rename("flagged_rows"))
display(p.loc[price_flags.any(axis=1)])

invalid_price         0
invalid_volume        0
invalid_OHLC_range    0
large_return          1
Name: flagged_rows, dtype: int64

,id,symbol,date,open,high,low,close,adj_close,volume
2411,2412,TSLA,2025-04-09,224.690002,274.690002,223.880005,272.200012,272.200012,219433400



**8. Check suspicious values.** Write plausible validity rules, then count and inspect exceptions across the full dataset.

- **Dates and coverage:** Check failed dates, minimum/maximum dates by company and table, unexpected gaps, and records outside your intended scope. Distinguish trading-day prices from calendar-day news and occasional filings. Verify what each date represents before interpreting alignment.
- **Prices and volume:** Investigate nonpositive prices, negative or non-whole volume, and inconsistent daily ranges. Should the low be no greater than the open/close and the high no less? Investigate large price changes. Do not force adjusted close inside the unadjusted daily range; the fields may use different adjustment bases.
- **Labels and text:** Review unexpected symbols or document types, empty or unusually short/long narratives, and any mismatch you notice between a text's dates or company references and its labels.

These are candidate checks, not evidence that such problems exist. An extreme observation may be real. Use supporting evidence before correcting or removing it; otherwise flag it and describe the limitation.


In [19]:
for name, frame in cleaned_tables.items():
    print(name)

    display(frame.groupby("symbol")["date"].agg(
        ["min", "max", "count"]
    ))

news_length = cleaned_tables["news"]["highlights"].str.len()

display(news_length.describe())
display(cleaned_tables["news"].loc[news_length.lt(100)])

for column in ["mda_content", "risk_content"]:
    print(column)
    display(cleaned_tables["filings"][column].str.len().describe())

prices


,min,max,count
symbol,,,
AAPL,2024-12-02,2026-03-31,332
ADBE,2024-12-02,2026-03-31,332
AMZN,2024-12-02,2026-03-31,332
GOOGL,2024-12-02,2026-03-31,332
META,2024-12-02,2026-03-31,332
MSFT,2024-12-02,2026-03-31,332
NVDA,2024-12-02,2026-03-31,332
TSLA,2024-12-02,2026-03-31,332


news


,min,max,count
symbol,,,
AAPL,2024-12-01,2026-03-31,486
ADBE,2024-12-01,2026-03-31,486
AMZN,2024-12-01,2026-03-31,486
GOOGL,2024-12-01,2026-03-31,486
META,2024-12-01,2026-03-31,486
MSFT,2024-12-01,2026-03-31,486
NVDA,2024-12-01,2026-03-31,486
TSLA,2024-12-01,2026-03-31,486


filings


,min,max,count
symbol,,,
AAPL,2024-02-01,2026-01-30,9
ADBE,2024-01-16,2026-03-25,10
AMZN,2024-02-01,2026-02-05,9
GOOGL,2024-01-30,2026-02-04,9
META,2024-02-01,2026-01-28,9
MSFT,2024-01-30,2026-01-28,9
NVDA,2024-02-21,2026-02-25,9
TSLA,2024-01-26,2026-01-28,9


... (2 outputs left out)

mda_content


count       73.000000
mean     43343.712329
std      13112.366360
min      15470.000000
25%      35479.000000
50%      46145.000000
75%      50613.000000
max      69271.000000
Name: mda_content, dtype: float64

risk_content


count        73.000000
mean      65818.356164
std       55929.630745
min         329.000000
25%       12283.000000
50%       65750.000000
75%       74886.000000
max      195820.000000
Name: risk_content, dtype: float64

### Step 6 — Combine tables only if your question needs it (checklist 9)

**9. Verify merges.** No merge is needed for the current price and volume metrics because all required columns are in prices. Merge-key and row-multiplication checks are not applicable to this calculation stage.

### Step 7 — Reconcile counts and check your metric (checklist 10–11)

**10. Reconcile changes.** Build a before/after audit for each table and any merged result. Explain every change in row count, key count, and important totals. Count removals at the step where they happen so overlapping problems are not counted twice. A flag or corrected cell does not necessarily remove a row.

In [20]:
audit_rows = []

for name in raw_tables:
    before = raw_tables[name]
    after = cleaned_tables[name]

    audit_rows.append({
        "table": name,
        "rows_before": len(before),
        "rows_after": len(after),
        "row_difference": len(after) - len(before),
        "keys_before": len(
            before[["symbol", "date"]].drop_duplicates()
        ),
        "keys_after": len(
            after[["symbol", "date"]].drop_duplicates()
        ),
    })

display(pd.DataFrame(audit_rows))

display(pd.DataFrame({
    "volume_before": (
        raw_tables["prices"].groupby("symbol")["volume"].sum()
    ),
    "volume_after": (
        cleaned_tables["prices"].groupby("symbol")["volume"].sum()
    ),
}))

,table,rows_before,rows_after,row_difference,keys_before,keys_after
0,prices,2656,2656,0,2656,2656
1,news,3888,3888,0,3888,3888
2,filings,73,73,0,73,73


,volume_before,volume_after
symbol,,
AAPL,17450834700,17450834700
ADBE,1454421600,1454421600
AMZN,14985388600,14985388600
GOOGL,11811858700,11811858700
META,5109326800,5109326800
MSFT,8217871100,8217871100
NVDA,70338418400,70338418400
TSLA,30088681000,30088681000



**11. Check calculations.** Define your chosen metric in words, its units, numerator/denominator where applicable, eligible records, and treatment of missing values. Report the contributing record count for **each metric and group**, including excluded records. State explicitly how you handle missing group keys and the difference between no observations and a genuine zero result.

**Hints:** If calculating returns, explain which price series you use, sort within company, check the previous observation's date, and handle the first observation or a missing/zero denominator. If comparing groups, consider whether their time coverage and sample sizes are comparable. For all-missing groups, verify that your result has the meaning you intended. Independently check a small calculation by hand.

**Fill in:** [Metric → eligible population → missing-value rule → missing-group-key rule → contributing count → independent check].

**Separate statuses and evidence for items 10 and 11:** [ ]. Add your own audit and calculations below.

In [21]:
p = cleaned_tables["prices"]

# Stop rather than silently exclude records with missing group/date keys
assert p[["symbol", "date"]].notna().all().all()

metrics = p.loc[
    p["symbol"].isin(analysis_symbols)
    & p["date"].between(start_date, end_date)
].copy()

metrics = metrics.sort_values(["symbol", "date"])

print("Rows before scope filter:", len(p))
print("Rows after scope filter:", len(metrics))
print("Rows excluded by scope:", len(p) - len(metrics))

group = metrics.groupby("symbol")

previous_close = group["adj_close"].shift(1)
previous_close = previous_close.where(previous_close.gt(0))

metrics["previous_date"] = group["date"].shift(1)

metrics["Return_Rate_1d"] = (
    metrics["adj_close"] / previous_close - 1
)

metrics["Return_Rate_3d"] = (
    group["adj_close"].shift(-2) / previous_close - 1
)

metrics["abs_return_1d"] = metrics["Return_Rate_1d"].abs()

metrics["intraday_range"] = (
    (metrics["high"] - metrics["low"])
    / metrics["open"].where(metrics["open"].gt(0))
)

metrics["average_volume_previous_20_days"] = (
    group["volume"].transform(
        lambda x: x.shift(1).rolling(20, min_periods=20).mean()
    )
)

volume_baseline = metrics["average_volume_previous_20_days"]

metrics["volume_ratio"] = (
    group["volume"].shift(-1)
    / volume_baseline.where(volume_baseline.gt(0))
)

metrics["PriceMA20"] = group["adj_close"].transform(
    lambda x: x.rolling(20, min_periods=20).mean()
)

display(metrics.head(25))

Rows before scope filter: 2656
Rows after scope filter: 2324
Rows excluded by scope: 332


,id,symbol,date,open,high,low,close,adj_close,volume,previous_date,Return_Rate_1d,Return_Rate_3d,abs_return_1d,intraday_range,average_volume_previous_20_days,volume_ratio,PriceMA20
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100,NaT,NaN,NaN,NaN,0.015299,NaN,NaN,NaN
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000,2024-12-02,0.012772,0.014400,0.012772,0.016096,NaN,NaN,NaN
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900,2024-12-03,0.001484,0.000783,0.001484,0.011776,NaN,NaN,NaN
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900,2024-12-04,0.000123,0.015390,0.000123,0.009877,NaN,NaN,NaN
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600,2024-12-05,-0.000823,0.019462,0.000823,0.010498,NaN,NaN,NaN
5,6,AAPL,2024-12-09,241.830002,247.240005,241.750000,246.750000,245.409409,44649200,2024-12-06,0.016101,0.015031,0.016101,0.022702,NaN,NaN,NaN
6,7,AAPL,2024-12-10,246.889999,248.210007,245.339996,247.770004,246.423859,36914800,2024-12-09,0.004134,0.004904,0.004134,0.011625,NaN,NaN,NaN
7,8,AAPL,2024-12-11,247.960007,250.800003,246.259995,246.490005,245.150818,45205800,2024-12-10,-0.005166,0.001453,0.005166,0.018309,NaN,NaN,NaN
8,9,AAPL,2024-12-12,246.889999,248.740005,245.679993,247.960007,246.612839,32777500,2024-12-11,0.005964,0.018459,0.005964,0.012394,NaN,NaN,NaN
9,10,AAPL,2024-12-13,247.820007,249.289993,246.240005,248.130005,246.781906,33155300,2024-12-12,0.000686,0.022262,0.000686,0.012307,NaN,NaN,NaN


### Step 8 — Verify five records and one Codex suggestion (checklist 12)

**12. Verify individual records.** Select five traceable records, including difficult cases found during inspection. Possibilities include formatting artifacts, a conversion problem, a duplicate candidate, an unmatched date, or an unusual price; use cases that actually exist. Include a normal record that should remain unchanged if helpful. Do not invent source problems just to fill categories.


In [22]:
p = cleaned_tables["prices"]

# Stop rather than silently exclude records with missing group/date keys
assert p[["symbol", "date"]].notna().all().all()

metrics = p.loc[
    p["symbol"].isin(analysis_symbols)
    & p["date"].between(start_date, end_date)
].copy()

metrics = metrics.sort_values(["symbol", "date"])

print("Rows before scope filter:", len(p))
print("Rows after scope filter:", len(metrics))
print("Rows excluded by scope:", len(p) - len(metrics))

group = metrics.groupby("symbol")

previous_close = group["adj_close"].shift(1)
previous_close = previous_close.where(previous_close.gt(0))

metrics["previous_date"] = group["date"].shift(1)

metrics["Return_Rate_1d"] = (
    metrics["adj_close"] / previous_close - 1
)

metrics["Return_Rate_3d"] = (
    group["adj_close"].shift(-2) / previous_close - 1
)

metrics["abs_return_1d"] = metrics["Return_Rate_1d"].abs()

metrics["intraday_range"] = (
    (metrics["high"] - metrics["low"])
    / metrics["open"].where(metrics["open"].gt(0))
)

metrics["average_volume_previous_20_days"] = (
    group["volume"].transform(
        lambda x: x.shift(1).rolling(20, min_periods=20).mean()
    )
)

volume_baseline = metrics["average_volume_previous_20_days"]

metrics["volume_ratio"] = (
    group["volume"].shift(-1)
    / volume_baseline.where(volume_baseline.gt(0))
)

metrics["PriceMA20"] = group["adj_close"].transform(
    lambda x: x.rolling(20, min_periods=20).mean()
)

display(metrics.head(25))

Rows before scope filter: 2656
Rows after scope filter: 2324
Rows excluded by scope: 332


,id,symbol,date,open,high,low,close,adj_close,volume,previous_date,Return_Rate_1d,Return_Rate_3d,abs_return_1d,intraday_range,average_volume_previous_20_days,volume_ratio,PriceMA20
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100,NaT,NaN,NaN,NaN,0.015299,NaN,NaN,NaN
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000,2024-12-02,0.012772,0.014400,0.012772,0.016096,NaN,NaN,NaN
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900,2024-12-03,0.001484,0.000783,0.001484,0.011776,NaN,NaN,NaN
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900,2024-12-04,0.000123,0.015390,0.000123,0.009877,NaN,NaN,NaN
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600,2024-12-05,-0.000823,0.019462,0.000823,0.010498,NaN,NaN,NaN
5,6,AAPL,2024-12-09,241.830002,247.240005,241.750000,246.750000,245.409409,44649200,2024-12-06,0.016101,0.015031,0.016101,0.022702,NaN,NaN,NaN
6,7,AAPL,2024-12-10,246.889999,248.210007,245.339996,247.770004,246.423859,36914800,2024-12-09,0.004134,0.004904,0.004134,0.011625,NaN,NaN,NaN
7,8,AAPL,2024-12-11,247.960007,250.800003,246.259995,246.490005,245.150818,45205800,2024-12-10,-0.005166,0.001453,0.005166,0.018309,NaN,NaN,NaN
8,9,AAPL,2024-12-12,246.889999,248.740005,245.679993,247.960007,246.612839,32777500,2024-12-11,0.005964,0.018459,0.005964,0.012394,NaN,NaN,NaN
9,10,AAPL,2024-12-13,247.820007,249.289993,246.240005,248.130005,246.781906,33155300,2024-12-12,0.000686,0.022262,0.000686,0.012307,NaN,NaN,NaN


In [23]:
metric_columns = [
    "adj_close", "Return_Rate_1d", "Return_Rate_3d",
    "abs_return_1d", "intraday_range", "volume",
    "volume_ratio", "PriceMA20",
]

valid_counts = metrics.groupby("symbol")[metric_columns].count()
total_counts = metrics.groupby("symbol").size()

display(valid_counts)
display(valid_counts.rsub(total_counts, axis=0))

,adj_close,Return_Rate_1d,Return_Rate_3d,abs_return_1d,intraday_range,volume,volume_ratio,PriceMA20
symbol,,,,,,,,
AAPL,332,331,329,331,332,332,311,313
AMZN,332,331,329,331,332,332,311,313
GOOGL,332,331,329,331,332,332,311,313
META,332,331,329,331,332,332,311,313
MSFT,332,331,329,331,332,332,311,313
NVDA,332,331,329,331,332,332,311,313
TSLA,332,331,329,331,332,332,311,313


,adj_close,Return_Rate_1d,Return_Rate_3d,abs_return_1d,intraday_range,volume,volume_ratio,PriceMA20
symbol,,,,,,,,
AAPL,0,1,3,1,0,0,21,19
AMZN,0,1,3,1,0,0,21,19
GOOGL,0,1,3,1,0,0,21,19
META,0,1,3,1,0,0,21,19
MSFT,0,1,3,1,0,0,21,19
NVDA,0,1,3,1,0,0,21,19
TSLA,0,1,3,1,0,0,21,19


In [24]:
example = metrics.loc[metrics["symbol"].eq("AAPL")].iloc[:2]

display(example[
    ["date", "adj_close", "previous_date", "Return_Rate_1d"]
])

manual_return = (
    example["adj_close"].iloc[1]
    / example["adj_close"].iloc[0] - 1
)

print("Direct calculation:", manual_return)
print("Calculated column:", example["Return_Rate_1d"].iloc[1])

,date,adj_close,previous_date,Return_Rate_1d
0,2024-12-02,238.288300,NaT,NaN
1,2024-12-03,241.331665,2024-12-02,0.012772


Direct calculation: 0.012771778908692744
Calculated column: 0.012771778908692744


In [25]:
record_checks = pd.DataFrame([
    ["filings", 1, "mda_content", "Decode HTML and trim outer whitespace"],
    ["filings", 66, "risk_content", "Keep the cross-reference; decode and trim"],
    ["news", 543, "highlights", "Keep the short text unchanged"],
    ["news", 1005, "highlights", "Keep the incomplete text unchanged"],
    ["prices", 4, "adj_close", "Keep the adjusted close unchanged"],
], columns=["table", "id", "column", "expected"])

display(record_checks)

,table,id,column,expected
0,filings,1,mda_content,Decode HTML and trim outer whitespace
1,filings,66,risk_content,Keep the cross-reference; decode and trim
2,news,543,highlights,Keep the short text unchanged
3,news,1005,highlights,Keep the incomplete text unchanged
4,prices,4,adj_close,Keep the adjusted close unchanged


In [26]:
comparisons = []

for _, check in record_checks.iterrows():
    before = raw_tables[check["table"]].set_index("id")
    after = cleaned_tables[check["table"]].set_index("id")

    original = before.loc[check["id"], check["column"]]
    actual = after.loc[check["id"], check["column"]]

    comparisons.append({
        "table": check["table"],
        "id": check["id"],
        "column": check["column"],
        "expected": check["expected"],
        "original": original,
        "actual": actual,
        "changed": original != actual,
    })

record_comparison = pd.DataFrame(comparisons)
display(record_comparison)

,table,id,column,expected,original,actual,changed
0,filings,1,mda_content,Decode HTML and trim outer whitespace,Item 2. Management&#8217;s Discussion and Ana...,Item 2. Management’s Discussion and Analysis o...,True
1,filings,66,risk_content,Keep the cross-reference; decode and trim,ITEM 1A. RISK FACTORS \n\nOur operations and ...,ITEM 1A. RISK FACTORS \n\nOur operations and f...,True
2,news,543,highlights,Keep the short text unchanged,The article also includes,The article also includes,False
3,news,1005,highlights,Keep the incomplete text unchanged,This item conveys a clear negative signal for ...,This item conveys a clear negative signal for ...,False
4,prices,4,adj_close,Keep the adjusted close unchanged,241.719559,241.719559,False


### Step 9 — Reproduce the work and explain limits (checklist 13–14)

**13. Test reproducibility.** Once your own implementation is complete, restart the kernel and run all cells from the unchanged original data in order. Check that the notebook works without variables from a previous session. Record the environment/dependencies, working directory, exact run steps, and expected output files. Explain how you compared regenerated outputs with the submitted results, including counts, schemas, and relevant values. Check that the raw files remain unchanged.


In [27]:
import sys
import pyarrow
import hashlib

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)
print("Working directory:", Path.cwd())

source_hashes_after = {}

for name in tables:
    path = DATA / f"{name}.parquet"
    source_hashes_after[name] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

hash_check = pd.DataFrame({
    "before": source_hashes_before,
    "after": source_hashes_after,
})

hash_check["unchanged"] = hash_check["before"].eq(
    hash_check["after"]
)

display(hash_check)

Python: 3.11.2 (v3.11.2:878ead1ac1, Feb  7 2023, 10:02:41) [Clang 13.0.0 (clang-1300.0.29.30)]
pandas: 3.0.5
pyarrow: 24.0.0
Working directory: /Users/yueyixin/dso576/Python-Group-Project/Evelyn_Herculean


,before,after,unchanged
prices,0001b983f1a302e2f7a2d2e6a6c407f44127de77e981b0...,0001b983f1a302e2f7a2d2e6a6c407f44127de77e981b0...,True
news,9156b73a5c417057104abe9eeb0d9b3ea0dceea66260e5...,9156b73a5c417057104abe9eeb0d9b3ea0dceea66260e5...,True
filings,2d9a93926e52d4e2f5f18e13d68fbdd60e14282f94df4e...,2d9a93926e52d4e2f5f18e13d68fbdd60e14282f94df4e...,True


In [28]:
for column in ["mda_content", "risk_content"]:
    once = cleaned_tables["filings"][column]
    twice = once.map(html.unescape).str.strip()

    print(column, "unchanged after second pass:", once.equals(twice))

mda_content unchanged after second pass: True
risk_content unchanged after second pass: True



**14. Document remaining limitations.** List unresolved quality issues and limitations in coverage or interpretation. Consider uncertain date meanings, missing publication times, incomplete text, unavailable records, excluded observations, or unequal coverage only where supported by your inspection. Explain how each could affect your business answer and what additional information would resolve it. Do not label an unresolved issue “Done” merely because you documented it.

In [29]:
limitations = pd.DataFrame({
    "issue": [
        "News has dates but no publication times",
        "Some news summaries appear short or incomplete",
        "Analysis includes only seven selected stocks",
        "Return_Rate_3d and volume_ratio use future observations",
    ],
    "effect": [
        "Same-day information availability is uncertain",
        "Text analysis may miss important context",
        "Results may not generalize to the wider market",
        "These metrics cannot be same-day known predictors",
    ],
    "handling": [
        "Document timing uncertainty",
        "Flag and retain the original text",
        "Report the selected stock universe",
        "Use as outcomes and retain boundary missing values",
    ],
})

display(limitations)

,issue,effect,handling
0,News has dates but no publication times,Same-day information availability is uncertain,Document timing uncertainty
1,Some news summaries appear short or incomplete,Text analysis may miss important context,Flag and retain the original text
2,Analysis includes only seven selected stocks,Results may not generalize to the wider market,Report the selected stock universe
3,Return_Rate_3d and volume_ratio use future obs...,These metrics cannot be same-day known predictors,Use as outcomes and retain boundary missing va...
